# SBI posterior parameter evaluation

Evaluates how accurately the NPE posterior recovers 25 physiological parameters from waveforms.

Uses the same 256-sim test batch as `cv-inverse-autoencoder/notebooks/autoencoder_param_eval_20260507.ipynb` for direct comparison.

**Point estimate comparison** (vs encoder): posterior mean vs GT — MAE, Median AE, MAPE  
**Posterior-specific**: 90% credible interval coverage per parameter

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from dataset import CVDataset, PARAM_KEYS, load_stats
from train_sbi import WaveformEmbedding  # needed to unpickle the posterior

In [ ]:
POSTERIOR_PATH = ROOT / 'outputs/exp_baseline_cnn4e64_maf5/posterior.pt'
STATS_PATH     = ROOT / 'norm_stats.json'
DATA_DIR       = Path('/media/8TBNVME/data/neh10/hdf5/cv8/simset_10M_cv8Eed_20260314/test')
MANIFEST       = DATA_DIR.parent / 'manifest_test.json'

BATCH_SIZE     = 256   # match autoencoder eval
BATCH_IDX      = 0
N_POSTERIOR_SAMPLES = 1000

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## Load posterior and test data

In [ ]:
posterior = torch.load(POSTERIOR_PATH, map_location=device, weights_only=False)
print('Posterior loaded:', type(posterior).__name__)

stats    = load_stats(STATS_PATH)
manifest = json.load(open(MANIFEST))

start  = BATCH_IDX * BATCH_SIZE
end    = start + BATCH_SIZE
entries = manifest['index'][start:end]
print(f'Test batch {BATCH_IDX}: sims {start}–{end - 1}  ({len(entries)} entries)')

ds     = CVDataset(str(DATA_DIR), entries, stats)
loader = DataLoader(ds, batch_size=BATCH_SIZE, num_workers=0)

all_theta, all_x = [], []
for theta_b, x_b in loader:
    all_theta.append(theta_b)
    all_x.append(x_b)
ds.close()

theta_gt = torch.cat(all_theta)  # (256, 25) — physical units
x_test   = torch.cat(all_x)      # (256, 5628)
print(f'GT params: {theta_gt.shape}  observations: {x_test.shape}')

## Sample posterior for each test observation

In [ ]:
# For each x_o draw N_POSTERIOR_SAMPLES — then summarise with mean and quantiles
all_means, all_q05, all_q95 = [], [], []

for i, x_o in enumerate(x_test):
    samples = posterior.sample((N_POSTERIOR_SAMPLES,), x=x_o.to(device), show_progress_bars=False)
    all_means.append(samples.mean(dim=0).cpu())
    all_q05.append(samples.quantile(0.05, dim=0).cpu())
    all_q95.append(samples.quantile(0.95, dim=0).cpu())
    if (i + 1) % 50 == 0:
        print(f'  {i + 1}/{len(x_test)} done')

pred_mean = torch.stack(all_means).numpy()  # (256, 25) — physical units
q05       = torch.stack(all_q05).numpy()
q95       = torch.stack(all_q95).numpy()
gt_phys   = theta_gt.numpy()

print(f'Posterior mean shape: {pred_mean.shape}')

In [ ]:
# Diagnostics — run before sampling
# 1. Check embedding output variance across test observations
emb_net = posterior.posterior_estimator.embedding_net

with torch.no_grad():
    emb_out = emb_net(x_test[:64].to(device))
print(f'Embedding output shape: {emb_out.shape}')
print(f'Per-dim std  min/mean/max: '
      f'{emb_out.std(0).min():.4f} / {emb_out.std(0).mean():.4f} / {emb_out.std(0).max():.4f}')
print(f'Overall mean: {emb_out.mean():.4f}  std: {emb_out.std():.4f}')

# 2. Sanity check: sample posterior on a TRAINING observation
train_manifest = json.load(open(MANIFEST.parent / 'manifest_train.json'))
ds_tr = CVDataset('/media/8TBNVME/data/neh10/hdf5/cv8/simset_10M_cv8Eed_20260314/train',
                  train_manifest['index'][:1], stats)
theta_tr, x_tr = ds_tr[0]
ds_tr.close()

with torch.no_grad():
    samples_tr = posterior.sample((500,), x=x_tr.to(device), show_progress_bars=False)
post_mean_tr = samples_tr.mean(0).cpu()

print(f"\nTraining obs sanity check:")
print(f"  {'param':<14} {'GT':>10} {'post mean':>12} {'err%':>8}")
for i, name in enumerate(PARAM_KEYS):
    err = abs(post_mean_tr[i].item() - theta_tr[i].item()) / (abs(theta_tr[i].item()) + 1e-9) * 100
    print(f"  {name:<14} {theta_tr[i].item():>10.4f} {post_mean_tr[i].item():>12.4f} {err:>7.1f}%")

## Point-estimate metrics (matches autoencoder eval)

In [ ]:
abs_err = np.abs(pred_mean - gt_phys)
ch_mae  = abs_err.mean(axis=0)
ch_med  = np.median(abs_err, axis=0)
ch_mape = (abs_err / (np.abs(gt_phys) + 1e-9) * 100).mean(axis=0)

print(f"{'Parameter':<14} {'MAE':>12} {'Median AE':>12} {'MAPE (%)':>10}")
print('-' * 52)
for i, name in enumerate(PARAM_KEYS):
    print(f'{name:<14} {ch_mae[i]:>12.4f} {ch_med[i]:>12.4f} {ch_mape[i]:>10.2f}%')

print(f'\nOverall  MAE (mean): {ch_mae.mean():.4f}  MAPE: {ch_mape.mean():.2f}%')

In [ ]:
# How much does the posterior mean actually vary across test observations?
post_mean_std = pred_mean.std(axis=0)   # std of posterior means across 256 samples
prior_range   = np.array([
    manifest['config']['pvar_high'][k] - manifest['config']['pvar_low'][k]
    for k in PARAM_KEYS
])
variation_pct = post_mean_std / prior_range * 100  # as % of prior range

print(f"{'param':<14} {'post-mean std':>14} {'prior range':>12} {'variation %':>12}")
print('-' * 55)
for i, name in enumerate(PARAM_KEYS):
    print(f"{name:<14} {post_mean_std[i]:>14.4f} {prior_range[i]:>12.4f} {variation_pct[i]:>11.1f}%")

print(f"\nMean variation across params: {variation_pct.mean():.1f}% of prior range")
print("(near 0% = flow ignoring x, ~10-30% = flow partially informed by x)")

In [ ]:
x = np.arange(len(PARAM_KEYS))
fig, axes = plt.subplots(1, 3, figsize=(22, 5))

axes[0].bar(x, ch_mae,  color='steelblue', alpha=0.85)
axes[1].bar(x, ch_med,  color='darkcyan',  alpha=0.85)
axes[2].bar(x, ch_mape, color='tomato',    alpha=0.85)

for ax, title, ylabel in zip(axes,
    ['Mean MAE per parameter', 'Median AE per parameter', 'MAPE per parameter'],
    ['AE (physical units)', 'AE (physical units)', 'MAPE (%)']):
    ax.set_xticks(x)
    ax.set_xticklabels(PARAM_KEYS, rotation=45, ha='right')
    ax.set_title(title)
    ax.set_ylabel(ylabel)

fig.suptitle('SBI posterior mean errors — exp_baseline_cnn4e64_maf5', fontsize=13)
plt.tight_layout()
plt.show()

## Predicted vs GT scatter — all 25 parameters

In [ ]:
ncols, nrows = 5, 5
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))

for i, name in enumerate(PARAM_KEYS):
    ax = axes[i // ncols, i % ncols]
    gt_i, pred_i = gt_phys[:, i], pred_mean[:, i]
    lo, hi = gt_i.min(), gt_i.max()
    ax.scatter(gt_i, pred_i, s=10, alpha=0.5, color='steelblue')
    ax.plot([lo, hi], [lo, hi], color='tomato', linewidth=1, linestyle='--')
    ax.set_title(f'{name}  (MAPE {ch_mape[i]:.1f}%)', fontsize=9)
    ax.set_xlabel('GT', fontsize=8)
    ax.set_ylabel('Posterior mean', fontsize=8)
    ax.tick_params(labelsize=7)

fig.suptitle('SBI — posterior mean vs GT params  (exp_baseline_cnn4e64_maf5)', fontsize=13)
plt.tight_layout()
plt.show()

## 90% credible interval coverage

For a well-calibrated posterior, ~90% of GT values should fall inside the 90% CI.

In [ ]:
in_ci = (gt_phys >= q05) & (gt_phys <= q95)  # (256, 25)
coverage = in_ci.mean(axis=0) * 100           # (25,) — % in 90% CI

print(f"{'Parameter':<14} {'90% CI coverage':>18}")
print('-' * 35)
for i, name in enumerate(PARAM_KEYS):
    flag = '  ✓' if 85 <= coverage[i] <= 95 else '  ←'
    print(f'{name:<14} {coverage[i]:>16.1f}%{flag}')

print(f'\nMean coverage: {coverage.mean():.1f}%  (target: 90%)')

In [ ]:
x = np.arange(len(PARAM_KEYS))
fig, ax = plt.subplots(figsize=(14, 4))
colors = ['steelblue' if 85 <= c <= 95 else 'tomato' for c in coverage]
ax.bar(x, coverage, color=colors, alpha=0.85)
ax.axhline(90, color='black', linestyle='--', linewidth=1, label='target 90%')
ax.axhline(85, color='gray',  linestyle=':',  linewidth=0.8)
ax.axhline(95, color='gray',  linestyle=':',  linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(PARAM_KEYS, rotation=45, ha='right')
ax.set_ylabel('Coverage (%)')
ax.set_title('90% credible interval coverage — exp_baseline_cnn4e64_maf5')
ax.legend()
plt.tight_layout()
plt.show()